In [ ]:
import logging
import sys
import warnings
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlflow.models.signature import infer_signature
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

logging.basicConfig(level=logging.WARN)
logger = logging.getLogger(__name__)

In [ ]:
# Load Data
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

# Split data into train and test sets(80, 20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Parameters
n_estimators = 100
max_depth = 10

In [ ]:
with mlflow.start_run():
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"Random Forest model (n_estimators={n_estimators}, max_depth={max_depth}):")
    print(f"RMSE: {rmse}")
    print(f"MAE: {mae}")
    print(f"R2: {r2}")

    signature = infer_signature(X_train, rf.predict(X_train))

    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
    
    # Manual Logging and Tagging
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("mae", mae)
    
    # Feature Importance Plot
    plt.figure(figsize=(10,6))
    feat_importances = pd.Series(rf.feature_importances_, index=X.columns)
    feat_importances.nlargest(10).plot(kind='barh')
    plt.title("Feature Importance")
    plt.tight_layout()
    
    # Save and Log Plot as Artifact
    plot_path = "feature_importance.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    
    # Log Model folder
    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(rf, "housing_model", registered_model_name="RandomForestCaliforniaHousing", signature=signature)
    else:
        mlflow.sklearn.log_model(rf, "housing_model", signature=signature)